In [1]:
import os
import numpy as np
import pandas as pd
from itertools import combinations, permutations
from scipy.stats import spearmanr, wilcoxon
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import LeaveOneGroupOut, StratifiedKFold

DATA_FILE = '../data/right.csv'
OUTPUT = '../results/'
os.makedirs(OUTPUT, exist_ok=True)

#taken from Subhash et.al, 2022
FEATURES = ['ax_mean', 'ax_var', 'ay_mean', 'ay_var', 'az_mean', 'az_var',
            'a_norm_mean', 'a_norm_var', 'a_norm_min', 'a_norm_max']

#three CV methods we evaluate
STRATEGIES = ['Random window-level', 'Contiguous block', 'Leave-one-task-out']

WINDOW = 0.25      # seconds
MIN_SAMPLES = 12   # 0.25 s at 50 Hz
N_FOLDS = 5
N_REPEATS = 10
SEED = 42

In [2]:
def window_features(end_time, window):
    functional = window.gnd.mean()
    return {'time': end_time,
            'subject': window.subject.iloc[0],
            'task': window.task.iloc[0],
            'gnd': np.nan if functional == 0.5 else int(functional > 0.5),
            'ax_mean': window.ax.mean(), 'ax_var': window.ax.var(),
            'ay_mean': window.ay.mean(), 'ay_var': window.ay.var(),
            'az_mean': window.az.mean(), 'az_var': window.az.var(),
            'a_norm_mean': window.a_norm.mean(), 'a_norm_var': window.a_norm.var(),
            'a_norm_min': window.a_norm.min(), 'a_norm_max': window.a_norm.max()}


def load_windows():
    raw = pd.read_csv(DATA_FILE, parse_dates=['time'])
    raw = raw.sort_values(['subject', 'time'])
    raw['task'] = raw.task.fillna('').str.strip()
    raw['a_norm'] = np.sqrt(raw.ax**2 + raw.ay**2 + raw.az**2)

    # new recording sequence at subject change, task change or a gap > 60 s
    gap = raw.time.diff().dt.total_seconds()
    new_segment = (raw.subject != raw.subject.shift()) | (raw.task != raw.task.shift()) | (gap > 60)

    rows = []
    for _, segment in raw.set_index('time').groupby(new_segment.cumsum().values):
        for start, window in segment.resample(str(WINDOW) + 's', origin='start'):
            if len(window) >= MIN_SAMPLES:
                rows.append(window_features(start + pd.Timedelta(seconds=WINDOW), window))

    windows = pd.DataFrame(rows).dropna(subset=['gnd'])
    windows = windows[windows.task != ''].astype({'gnd': int})
    return windows

In [3]:
def shuffled_splits(subject):
    for repeat in range(N_REPEATS):
        folds = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED + repeat)
        for train, test in folds.split(subject, subject.gnd):
            yield repeat, train, test


def time_blocks(n):
    blocks = np.array_split(np.arange(n), N_FOLDS)
    return np.concatenate([np.full(len(b), k) for k, b in enumerate(blocks)])


def contiguous_splits(subject):
    # run = consecutive windows of one task with no break in between;
    # each run is divided into five blocks of consecutive windows
    # (a run with fewer than five windows gives one window per block)
    gap = subject.time.diff().dt.total_seconds()
    run = ((subject.task != subject.task.shift()) | (gap > WINDOW)).cumsum()
    block = run.groupby(run).transform(lambda r: time_blocks(len(r)))
    for fold in range(N_FOLDS):
        yield 0, np.flatnonzero(block != fold), np.flatnonzero(block == fold)


def task_out_splits(subject):
    for train, test in LeaveOneGroupOut().split(subject, groups=subject.task):
        yield 0, train, test

In [4]:
def run_cv(windows, strategy, splits):
    results = []
    for _, subject in windows.groupby('subject'):
        subject = subject.sort_values('time').reset_index(drop=True)
        for repeat, train, test in splits(subject):
            model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=SEED, n_jobs=-1)
            model.fit(subject.loc[train, FEATURES], subject.gnd[train])
            result = subject.loc[test, ['subject', 'gnd']].assign(strategy=strategy, repeat=repeat)
            result['prediction'] = model.predict(subject.loc[test, FEATURES])
            results.append(result)
    return pd.concat(results)


def scores(result):
    tn, fp, fn, tp = confusion_matrix(result.gnd, result.prediction, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)
    youden = sensitivity + specificity - 1
    return pd.Series({'sensitivity': sensitivity,
                      'specificity': specificity,
                      'youden_index': youden,
                      'ppv': tp / (tp + fp),
                      'npv': tn / (tn + fn),
                      'balanced_accuracy': (1 + youden) / 2})

In [5]:
windows = load_windows()

# exclude subjects with fewer than 5 windows in any task
counts = windows.groupby(['subject', 'task']).size().unstack(fill_value=0)
keep = counts.index[(counts >= N_FOLDS).all(axis=1)]
print('excluded:', list(counts.index.difference(keep)))
windows = windows[windows.subject.isin(keep)]

predictions = pd.concat([run_cv(windows, STRATEGIES[0], shuffled_splits),
                         run_cv(windows, STRATEGIES[1], contiguous_splits),
                         run_cv(windows, STRATEGIES[2], task_out_splits)])

excluded: [3, 5, 6]


In [6]:
# metrics from pooled predictions per repeat (10 for random, 1 for others), then averaged per subject
repeat_scores = predictions.groupby(['strategy', 'subject', 'repeat'])[['gnd', 'prediction']].apply(scores)
subject_scores = repeat_scores.groupby(['strategy', 'subject']).mean()

summary = subject_scores.groupby('strategy').agg(['mean', 'std']).loc[STRATEGIES]
summary.round(3).to_csv(OUTPUT + 'performance_summary.csv')
summary.round(3)

sensitivity        specificity        youden_index         \
                           mean    std        mean    std         mean    std   
strategy                                                                        
Random window-level       0.997  0.002       0.888  0.064        0.885  0.065   
Contiguous block          0.994  0.006       0.873  0.064        0.867  0.068   
Leave-one-task-out        0.982  0.024       0.636  0.214        0.618  0.208   

                       ppv           npv        balanced_accuracy         
                      mean    std   mean    std              mean    std  
strategy                                                                  
Random window-level  0.982  0.013  0.973  0.018             0.942  0.033  
Contiguous block     0.979  0.014  0.962  0.033             0.934  0.034  
Leave-one-task-out   0.952  0.027  0.879  0.108             0.809  0.104

In [7]:
functional = (100 * windows.groupby('subject').gnd.mean()).rename('functional_percent')
by_subject = subject_scores.reset_index().merge(functional, on='subject')
by_subject.round(3).to_csv(OUTPUT + 'performance_by_subject.csv', index=False)
by_subject.round(3)

,strategy,subject,sensitivity,specificity,youden_index,ppv,npv,balanced_accuracy,functional_percent
0,Contiguous block,1,0.999,0.849,0.848,0.971,0.993,0.924,83.367
1,Contiguous block,2,0.995,0.809,0.803,0.976,0.949,0.902,88.867
2,Contiguous block,4,0.999,0.946,0.945,0.987,0.995,0.973,80.895
3,Contiguous block,7,0.999,0.956,0.955,0.997,0.985,0.977,93.639
4,Contiguous block,8,0.994,0.909,0.903,0.983,0.966,0.952,84.139
5,Contiguous block,9,0.996,0.840,0.836,0.987,0.940,0.918,92.308
6,Contiguous block,10,0.981,0.799,0.780,0.955,0.906,0.890,81.250
7,Leave-one-task-out,1,0.975,0.825,0.800,0.965,0.867,0.900,83.367
8,Leave-one-task-out,2,0.986,0.678,0.664,0.961,0.857,0.832,88.867
9,Leave-one-task-out,4,0.999,0.927,0.926,0.983,0.995,0.963,80.895


In [8]:
youden = subject_scores.youden_index.unstack('strategy')

tests = []
for a, b in combinations(STRATEGIES, 2):
    diff = youden[a] - youden[b]
    test = wilcoxon(youden[a], youden[b], method='exact')   # W = min(W+, W-), two-sided
    tests.append({'comparison': a + ' vs ' + b,
                  'median_diff': diff.median(),
                  'iqr_low': diff.quantile(0.25),
                  'iqr_high': diff.quantile(0.75),
                  'W': test.statistic,
                  'p': test.pvalue,
                  'p_bonferroni': min(3 * test.pvalue, 1)})

tests = pd.DataFrame(tests)
tests.round(3).to_csv(OUTPUT + 'wilcoxon_tests.csv', index=False)
tests.round(3)

,comparison,median_diff,iqr_low,iqr_high,W,p,p_bonferroni
0,Random window-level vs Contiguous block,0.008,-0.002,0.025,7.0,0.297,0.891
1,Random window-level vs Leave-one-task-out,0.253,0.131,0.355,0.0,0.016,0.047
2,Contiguous block vs Leave-one-task-out,0.245,0.094,0.330,0.0,0.016,0.047


In [9]:
# Spearman correlation: functional % vs leave-one-task-out Youden index,
# exact two-sided p-value from all 7! orderings
x = functional.loc[youden.index]
y = youden['Leave-one-task-out']
rho = spearmanr(x, y).statistic
null = [spearmanr(x, list(order)).statistic for order in permutations(y)]
p_exact = np.mean(np.abs(null) >= abs(rho) - 1e-12)
print(f'Spearman rho = {rho:.2f}, exact two-sided p = {p_exact:.3f}')

Spearman rho = -0.75, exact two-sided p = 0.066


In [10]:
# task containing most non-functional windows per subject and its share
nonfunctional = windows[windows.gnd == 0].groupby(['subject', 'task']).size()
rows = []
for subject, counts in nonfunctional.groupby(level='subject'):
    counts = counts.droplevel('subject')
    rows.append({'subject': subject,
                 'task': counts.idxmax(),
                 'percent_of_nonfunctional': 100 * counts.max() / counts.sum()})

top_task = pd.DataFrame(rows)
top_task.round(1).to_csv(OUTPUT + 'nonfunctional_by_task.csv', index=False)
top_task.round(1)

,subject,task,percent_of_nonfunctional
0,1,Walk25,68.1
1,2,Walk25,60.0
2,4,Walk25,82.9
3,7,Walk25,92.6
4,8,Walk25,44.9
5,9,Walk25,74.5
6,10,CallMobile,43.6
